# Unidade 3 - Bloco prático da Aula 01: TPE vs Random no Optuna

Compara o RandomSampler e o TPESampler do Optuna com o mesmo orçamento de 40 tentativas sobre um gradient boosting. Além do melhor resultado único, olhe a média das últimas tentativas de cada método: é ali que a estratégia do TPE aparece.

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 21.0 MB/s eta 0:00:00


In [4]:
import time
import numpy as np
import optuna

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold


# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

optuna.logging.set_verbosity(optuna.logging.WARNING)

X, y = load_breast_cancer(return_X_y=True)

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

N_TRIALS = 30


# ============================================================
# 2. FUNÇÃO OBJETIVO
# ============================================================

def objetivo(trial):

    # Optuna escolhe os hiperparâmetros
    parametros = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            50,
            300
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            5
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        )
    }

    # Criação do modelo
    modelo = GradientBoostingClassifier(
        **parametros,

        # Early stopping
        n_iter_no_change=10,
        validation_fraction=0.15,
        tol=1e-4,

        random_state=42
    )

    # Validação cruzada
    scores = cross_val_score(
        modelo,
        X,
        y,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    # Média dos 3 folds
    return scores.mean()


# ============================================================
# 3. CALLBACK PARA MOSTRAR O PROCESSO
# ============================================================

def mostrar_trial(study, trial):

    print(
        f"Trial {trial.number + 1:02d} | "
        f"F1 = {trial.value:.4f} | "
        f"Melhor = {study.best_value:.4f}"
    )

    print(
        f"    parâmetros = {trial.params}"
    )


# ============================================================
# 4. SAMPLERS QUE SERÃO COMPARADOS
# ============================================================

samplers = {
    "Random": optuna.samplers.RandomSampler(
        seed=42
    ),

    "TPE": optuna.samplers.TPESampler(
        seed=42
    )
}


# ============================================================
# 5. EXECUÇÃO DOS EXPERIMENTOS
# ============================================================

estudos = {}

for nome, sampler in samplers.items():

    print("\n" + "=" * 70)
    print(f"INICIANDO OTIMIZAÇÃO COM {nome}")
    print("=" * 70)

    inicio = time.time()

    estudo = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    estudo.optimize(
        objetivo,
        n_trials=N_TRIALS,
        callbacks=[mostrar_trial]
    )

    tempo = time.time() - inicio

    estudos[nome] = estudo

    print("\nRESULTADO FINAL")

    print(
        f"Melhor F1 = {estudo.best_value:.4f}"
    )

    print(
        f"Melhores parâmetros = {estudo.best_params}"
    )

    print(
        f"Tempo total = {tempo:.2f} segundos"
    )


# ============================================================
# 6. COMPARAÇÃO RANDOM x TPE
# ============================================================

print("\n" + "=" * 70)
print("EVOLUÇÃO DOS MÉTODOS")
print("=" * 70)

for nome, estudo in estudos.items():

    valores = [
        trial.value
        for trial in estudo.trials
        if trial.value is not None
    ]

    print(f"\n{nome}")

    for marco in [5, 10, 20, N_TRIALS]:

        if marco <= len(valores):

            melhor = max(
                valores[:marco]
            )

            print(
                f"Após {marco:02d} trials → "
                f"melhor F1 = {melhor:.4f}"
            )


# ============================================================
# 7. IMPORTÂNCIA DOS HIPERPARÂMETROS
# ============================================================

print("\n" + "=" * 70)
print("IMPORTÂNCIA DOS HIPERPARÂMETROS DO TPE")
print("=" * 70)

importancias = optuna.importance.get_param_importances(
    estudos["TPE"]
)

for parametro, importancia in importancias.items():

    print(
        f"{parametro:15s} → "
        f"{importancia:.2%}"
    )


INICIANDO OTIMIZAÇÃO COM Random
Trial 01 | F1 = 0.9601 | Melhor = 0.9601
    parâmetros = {'n_estimators': 144, 'learning_rate': 0.2536999076681772, 'max_depth': 4, 'subsample': 0.8394633936788146}
Trial 02 | F1 = 0.9497 | Melhor = 0.9601
    parâmetros = {'n_estimators': 89, 'learning_rate': 0.01699897838270077, 'max_depth': 2, 'subsample': 0.9464704583099741}
Trial 03 | F1 = 0.9575 | Melhor = 0.9601
    parâmetros = {'n_estimators': 200, 'learning_rate': 0.11114989443094977, 'max_depth': 2, 'subsample': 0.9879639408647978}
Trial 04 | F1 = 0.9561 | Melhor = 0.9601
    parâmetros = {'n_estimators': 258, 'learning_rate': 0.020589728197687916, 'max_depth': 2, 'subsample': 0.6733618039413735}
Trial 05 | F1 = 0.9613 | Melhor = 0.9613
    parâmetros = {'n_estimators': 126, 'learning_rate': 0.05958389350068958, 'max_depth': 3, 'subsample': 0.7164916560792167}
Trial 06 | F1 = 0.9559 | Melhor = 0.9613
    parâmetros = {'n_estimators': 203, 'learning_rate': 0.01607123851203988, 'max_depth': 3,